# Módulos estándar · os y sys

El sistema de ficheros, el entorno y el propio intérprete vistos desde Python: carpetas, rutas y tamaños con `os` y `os.path` (comparados con `pathlib`), recorridos con `os.walk`, variables de entorno, procesos hijos con `subprocess`, argumentos y códigos de salida con `sys.argv` y `sys.exit`, `sys.path` y el límite de recursión. Todo se hace dentro de la carpeta `pruebas_os/`, que el notebook crea junto a sí y borra en el último ejercicio. Ideas de [PYnative](https://pynative.com/python-exercises-with-solutions/), [machinelearningplus](https://www.machinelearningplus.com/) y [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios).

## 1. Carpeta de trabajo

Mostrar el nombre de la carpeta actual y crear dentro la estructura `pruebas_os/datos/2026`, `pruebas_os/fotos` y `pruebas_os/docs` con `os.makedirs`. Comprobar que repetir la llamada con `exist_ok=True` no falla y que sin esa opción se lanza `FileExistsError`.

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

CARPETA = "pruebas_os"

print("Carpeta actual:", os.path.basename(os.getcwd()))
for sub in (("datos", "2026"), ("fotos",), ("docs",)):
    os.makedirs(os.path.join(CARPETA, *sub), exist_ok=True)
os.makedirs(os.path.join(CARPETA, "datos", "2026"), exist_ok=True)
try:
    os.makedirs(os.path.join(CARPETA, "datos"))
except FileExistsError as e:
    print("Sin exist_ok:", type(e).__name__)

print("Contenido de pruebas_os:", sorted(os.listdir(CARPETA)))
assert os.path.isabs(os.getcwd())
assert os.path.isdir(CARPETA) and not os.path.isfile(CARPETA)
assert not os.path.exists(os.path.join(CARPETA, "no_existe"))

Carpeta actual: 10-modulos-estandar
Sin exist_ok: FileExistsError
Contenido de pruebas_os: ['datos', 'docs', 'fotos']


## 2. Ficheros, extensiones y tamaños

Crear en `pruebas_os/` varios ficheros de tamaño conocido (incluidos uno oculto, uno con dos puntos en el nombre y uno dentro de una carpeta `.papelera`). Después, listar el contenido de `datos/` indicando si cada entrada es carpeta o fichero, su nombre sin extensión, la extensión (`os.path.splitext`) y el tamaño (`os.path.getsize`).

In [2]:
FICHEROS = {
    "datos/ventas.csv": 1_500,
    "datos/clientes.csv": 2_048,
    "datos/LEEME.txt": 300,
    "datos/2026/enero.csv": 4_096,
    "datos/2026/febrero.csv": 5_000,
    "fotos/teide.jpg": 1_572_864,
    "fotos/roque-nublo.JPG": 800_000,
    "docs/informe.final.pdf": 120_000,
    "docs/.oculto": 10,
    "docs/.papelera/borrador.pdf": 50_000,
}
for ruta, tamaño in FICHEROS.items():
    completa = os.path.join(CARPETA, *ruta.split("/"))
    os.makedirs(os.path.dirname(completa), exist_ok=True)
    with open(completa, "wb") as f:
        f.write(b"\0" * tamaño)

datos = os.path.join(CARPETA, "datos")
print(f"{'Entrada':<14} {'Tipo':<8} {'Nombre':<9} {'Ext.':<5} {'Bytes':>6}")
for nombre in sorted(os.listdir(datos)):
    ruta = os.path.join(datos, nombre)
    raiz, ext = os.path.splitext(nombre)
    if os.path.isdir(ruta):
        print(f"{nombre:<14} {'carpeta':<8}")
    else:
        print(f"{nombre:<14} {'fichero':<8} {raiz:<9} {ext:<5} {os.path.getsize(ruta):>6}")

Entrada        Tipo     Nombre    Ext.   Bytes
2026           carpeta 
LEEME.txt      fichero  LEEME     .txt     300
clientes.csv   fichero  clientes  .csv    2048
ventas.csv     fichero  ventas    .csv    1500


`splitext` solo separa la última extensión y no considera extensión el punto inicial de un fichero oculto; `Path.suffix` hace lo mismo.

In [3]:
for nombre in ("informe.final.pdf", ".oculto", "roque-nublo.JPG", "sin_extension"):
    print(f"{nombre:<18} {os.path.splitext(nombre)!s:<28} suffix={Path(nombre).suffix!r}")
    assert os.path.splitext(nombre)[1] == Path(nombre).suffix

assert Path("informe.final.pdf").suffixes == [".final", ".pdf"]
assert os.path.getsize(os.path.join(CARPETA, "fotos", "teide.jpg")) == 1_572_864

informe.final.pdf  ('informe.final', '.pdf')    suffix='.pdf'
.oculto            ('.oculto', '')              suffix=''
roque-nublo.JPG    ('roque-nublo', '.JPG')      suffix='.JPG'
sin_extension      ('sin_extension', '')        suffix=''


## 3. Tamaño de un árbol con `os.walk`

Escribir `legible(n)`, que muestre un número de bytes en B, KB, MB o GB (con potencias de 1024), y recorrer `pruebas_os/` con `os.walk` dibujando el árbol de carpetas con el tamaño de los ficheros de cada una y el total. Ordenar la lista de subcarpetas en su sitio para que el recorrido salga siempre en el mismo orden, y comprobar el total con `pathlib`.

In [4]:
def legible(n):
    for unidad in ("B", "KB", "MB"):
        if n < 1024:
            return f"{n} {unidad}" if unidad == "B" else f"{n:.1f} {unidad}"
        n /= 1024
    return f"{n:.1f} GB"


def tamaño_arbol(raiz, mostrar=False):
    total = 0
    for carpeta, subcarpetas, nombres in os.walk(raiz):
        # os.walk visita las subcarpetas en el orden de esta lista: ordenarla en su sitio fija el recorrido
        subcarpetas.sort()
        propio = sum(os.path.getsize(os.path.join(carpeta, n)) for n in nombres)
        total += propio
        if mostrar:
            nivel = len(Path(carpeta).relative_to(raiz).parts)
            print(f"{'    ' * nivel + os.path.basename(carpeta) + '/':<22} {len(nombres)} fich. {legible(propio):>10}")
    return total


total = tamaño_arbol(CARPETA, mostrar=True)
print("Total:", legible(total))

con_pathlib = sum(p.stat().st_size for p in Path(CARPETA).rglob("*") if p.is_file())
assert total == con_pathlib == sum(FICHEROS.values())
assert [legible(300), legible(2_048), legible(1_572_864), legible(3 * 1024**3)] == ["300 B", "2.0 KB", "1.5 MB", "3.0 GB"]

pruebas_os/            0 fich.        0 B
    datos/             3 fich.     3.8 KB
        2026/          2 fich.     8.9 KB
    docs/              2 fich.   117.2 KB
        .papelera/     1 fich.    48.8 KB
    fotos/             2 fich.     2.3 MB
Total: 2.4 MB


## 4. Buscar ficheros por extensión

Escribir `buscar(raiz, *extensiones)`, que devuelva las rutas relativas (con `/`) de los ficheros con alguna de esas extensiones sin distinguir mayúsculas, saltándose las carpetas ocultas (las que empiezan por punto). Comprobarlo con `Path.rglob`, que sí entra en las carpetas ocultas.

In [5]:
def buscar(raiz, *extensiones):
    extensiones = {e.lower() for e in extensiones}
    encontrados = []
    for carpeta, subcarpetas, nombres in os.walk(raiz):
        # Cambiar la lista en su sitio (no reasignarla) es lo que impide que os.walk entre en esas carpetas
        subcarpetas[:] = [s for s in subcarpetas if not s.startswith(".")]
        for nombre in nombres:
            if os.path.splitext(nombre)[1].lower() in extensiones:
                encontrados.append(Path(os.path.relpath(os.path.join(carpeta, nombre), raiz)).as_posix())
    return sorted(encontrados)


for exts in ((".csv",), (".jpg",), (".pdf", ".txt")):
    print(f"{' '.join(exts):<10} {buscar(CARPETA, *exts)}")

.csv       ['datos/2026/enero.csv', 'datos/2026/febrero.csv', 'datos/clientes.csv', 'datos/ventas.csv']
.jpg       ['fotos/roque-nublo.JPG', 'fotos/teide.jpg']
.pdf .txt  ['datos/LEEME.txt', 'docs/informe.final.pdf']


In [6]:
def con_rglob(raiz, *extensiones):
    return sorted(p.relative_to(raiz).as_posix() for p in Path(raiz).rglob("*")
                  if p.is_file() and p.suffix.lower() in extensiones)


assert buscar(CARPETA, ".csv") == con_rglob(CARPETA, ".csv")
assert buscar(CARPETA, ".JPG") == con_rglob(CARPETA, ".jpg") == ["fotos/roque-nublo.JPG", "fotos/teide.jpg"]
print("rglob también ve:", sorted(set(con_rglob(CARPETA, ".pdf")) - set(buscar(CARPETA, ".pdf"))))
# Con case_sensitive=False el patrón ignora mayúsculas también en Linux y macOS
assert len(list(Path(CARPETA).rglob("*.jpg", case_sensitive=False))) == 2

rglob también ve: ['docs/.papelera/borrador.pdf']


## 5. Rutas portables

Construir la ruta de `pruebas_os/datos/2026/enero.csv` con `os.path.join` y con `Path`, y comprobar que coinciden. Obtener su ruta absoluta (sin mostrarla, porque depende del equipo), su carpeta, nombre, raíz y extensión, la ruta relativa de `fotos/teide.jpg` vista desde `datos/2026` y la forma normalizada de una ruta con `.` y `..`.

In [7]:
ruta = os.path.join(CARPETA, "datos", "2026", "enero.csv")
p = Path(CARPETA) / "datos" / "2026" / "enero.csv"
print("Separador del sistema:", repr(os.sep))
print("os.path.join:", ruta)
print("Path (con /):", p.as_posix())
assert str(p) == ruta

absoluta = os.path.abspath(ruta)
assert os.path.isabs(absoluta) and absoluta.endswith(ruta)
assert Path(absoluta) == p.resolve()

print(f"carpeta={os.path.basename(os.path.dirname(ruta))!r} nombre={os.path.basename(ruta)!r} "
      f"raíz={p.stem!r} extensión={p.suffix!r} partes={p.parts}")

desde = os.path.join(CARPETA, "datos", "2026")
relativa = os.path.relpath(os.path.join(CARPETA, "fotos", "teide.jpg"), desde)
print("teide.jpg desde datos/2026:", Path(relativa).as_posix())
assert os.path.exists(os.path.join(desde, relativa))

rara = os.path.join(CARPETA, "datos", ".", "2026", "..", "ventas.csv")
print("normpath:", Path(os.path.normpath(rara)).as_posix())
assert os.path.normpath(rara) == os.path.join(CARPETA, "datos", "ventas.csv")
assert p.with_suffix(".json").name == "enero.json" and p.parent.name == "2026"

Separador del sistema: '\\'
os.path.join: pruebas_os\datos\2026\enero.csv
Path (con /): pruebas_os/datos/2026/enero.csv
carpeta='2026' nombre='enero.csv' raíz='enero' extensión='.csv' partes=('pruebas_os', 'datos', '2026', 'enero.csv')
teide.jpg desde datos/2026: ../../fotos/teide.jpg
normpath: pruebas_os/datos/ventas.csv


## 6. Variables de entorno

Leer una variable que no existe con un valor por defecto, crear tres variables con el prefijo `CANARIAS_` y listarlas ordenadas filtrando por ese prefijo. Comprobar que los valores tienen que ser cadenas y borrar las variables al terminar.

In [8]:
print("ISLA_FAVORITA:", os.environ.get("ISLA_FAVORITA", "La Gomera (por defecto)"))
print("getenv sin defecto:", os.getenv("ISLA_FAVORITA"))

os.environ["CANARIAS_ISLA"] = "Tenerife"
os.environ["CANARIAS_MUNICIPIO"] = "La Orotava"
os.environ["CANARIAS_ALTITUD"] = "3715"
try:
    os.environ["CANARIAS_ALTITUD"] = 3715
except TypeError as e:
    print("Valor no textual:", e)

propias = {k: v for k, v in sorted(os.environ.items()) if k.startswith("CANARIAS_")}
for clave, valor in propias.items():
    print(f"  {clave:<20} = {valor}")
assert list(propias) == ["CANARIAS_ALTITUD", "CANARIAS_ISLA", "CANARIAS_MUNICIPIO"]
assert int(os.environ["CANARIAS_ALTITUD"]) + 1 == 3716

for clave in propias:
    del os.environ[clave]
assert not any(k.startswith("CANARIAS_") for k in os.environ)

ISLA_FAVORITA: La Gomera (por defecto)
getenv sin defecto: None
Valor no textual: str expected, not int
  CANARIAS_ALTITUD     = 3715
  CANARIAS_ISLA        = Tenerife
  CANARIAS_MUNICIPIO   = La Orotava


## 7. Configuración desde el entorno

Escribir `cargar_config(entorno)`, que parta de unos valores por defecto (puerto, modo de depuración, isla y tipo del IGIC) y los sobrescriba con las variables `APP_PUERTO`, `APP_DEPURAR`, etc. que haya en `entorno`, convirtiendo cada texto al tipo del valor por defecto. Un valor que no se pueda convertir debe dar un error claro. Probarlo con diccionarios y con el `os.environ` real.

In [9]:
POR_DEFECTO = {"puerto": 8000, "depurar": False, "isla": "Gran Canaria", "igic": 0.07}


def a_bool(texto):
    t = texto.strip().lower()
    if t in {"1", "true", "sí", "si", "on"}:
        return True
    if t in {"0", "false", "no", "off", ""}:
        return False
    raise ValueError(f"no es un booleano: {texto!r}")


def cargar_config(entorno=os.environ, prefijo="APP_"):
    config = dict(POR_DEFECTO)
    for clave, defecto in POR_DEFECTO.items():
        texto = entorno.get(prefijo + clave.upper())
        if texto is None:
            continue
        # bool("false") es True: los booleanos necesitan su propio conversor
        conversor = a_bool if isinstance(defecto, bool) else type(defecto)
        try:
            config[clave] = conversor(texto)
        except ValueError:
            raise ValueError(f"{prefijo}{clave.upper()}={texto!r} no es un {type(defecto).__name__}") from None
    return config


entorno = {"APP_PUERTO": "8080", "APP_DEPURAR": "sí", "OTRA": "x"}
config = cargar_config(entorno)
for clave, valor in config.items():
    origen = "entorno" if "APP_" + clave.upper() in entorno else "defecto"
    print(f"{clave:<8} {valor!r:<15} ({origen})")
assert config == {"puerto": 8080, "depurar": True, "isla": "Gran Canaria", "igic": 0.07}

try:
    cargar_config({"APP_PUERTO": "ocho mil"})
except ValueError as e:
    print("Error:", e)

os.environ["APP_ISLA"] = "Lanzarote"
os.environ["APP_DEPURAR"] = "off"
try:
    assert cargar_config() == POR_DEFECTO | {"isla": "Lanzarote"}
finally:
    del os.environ["APP_ISLA"], os.environ["APP_DEPURAR"]
assert cargar_config() == POR_DEFECTO

puerto   8080            (entorno)
depurar  True            (entorno)
isla     'Gran Canaria'  (defecto)
igic     0.07            (defecto)
Error: APP_PUERTO='ocho mil' no es un int


## 8. Proceso, intérprete y sistema

Obtener el identificador del proceso actual y el de su padre, y comprobar que un proceso hijo lanzado con `subprocess` tiene otro distinto. Mostrar solo datos estables del intérprete y del sistema (versión mayor de Python, implementación y sistema operativo) y comprobar con `assert` que `sys` y `platform` dicen lo mismo.

In [10]:
import platform

pid, ppid = os.getpid(), os.getppid()
hijo = subprocess.run([sys.executable, "-c", "import os; print(os.getpid())"], capture_output=True, text=True)
assert pid > 0 and ppid > 0 and pid != ppid
assert int(hijo.stdout) != pid
print("El hijo tiene otro PID:", int(hijo.stdout) != pid)

print(f"Python {sys.version_info.major} ({platform.python_implementation()}) en {platform.system()}")
assert sys.version_info >= (3, 10)
assert sys.version_info[:3] == tuple(map(int, platform.python_version_tuple()))
assert (sys.platform == "win32") == (platform.system() == "Windows")
print("Enteros nativos de 64 bits:", sys.maxsize == 2**63 - 1)

if sys.version_info >= (3, 11):
    import tomllib
    print("tomllib disponible:", tomllib.loads('isla = "El Hierro"'))

El hijo tiene otro PID: True
Python 3 (CPython) en Windows
Enteros nativos de 64 bits: True
tomllib disponible: {'isla': 'El Hierro'}


## 9. Ejecutar comandos con `subprocess`

Ejecutar otro intérprete de Python con `subprocess.run` y capturar su salida, ver el código de salida de un programa que termina con `sys.exit(3)`, convertirlo en excepción con `check=True`, cortar con `timeout` un programa que tarda demasiado y pasarle al hijo una variable de entorno con `env`. Se usa `sys.executable` en lugar de órdenes como `dir` o `ls`, que dependen del sistema.

In [11]:
def ejecutar(*argumentos, **opciones):
    """Ejecuta Python con UTF-8 en las dos direcciones, para que las tildes lleguen bien en cualquier sistema."""
    entorno = os.environ | {"PYTHONIOENCODING": "utf-8"} | opciones.pop("extra_env", {})
    return subprocess.run([sys.executable, *argumentos], capture_output=True, encoding="utf-8",
                          env=entorno, **opciones)


r = ejecutar("-c", "print('Hola desde otro proceso')")
print(f"código {r.returncode}: {r.stdout.strip()}")

r = ejecutar("-c", "import sys; print('a medias'); sys.exit(3)")
print(f"código {r.returncode}: {r.stdout.strip()}")
assert r.returncode == 3

try:
    ejecutar("-c", "import sys; sys.exit(3)", check=True)
except subprocess.CalledProcessError as e:
    print("check=True:", type(e).__name__, "con código", e.returncode)

try:
    # Sin capturar la salida: así no se espera a que se cierren las tuberías tras matar el proceso
    subprocess.run([sys.executable, "-c", "import time; time.sleep(5)"], timeout=0.5)
except subprocess.TimeoutExpired as e:
    print(f"timeout: cortado a los {e.timeout} s")

r = ejecutar("-c", "import os; print(os.environ['CANARIAS_SALUDO'])", extra_env={"CANARIAS_SALUDO": "¡Buenas, mi niño!"})
print("El hijo recibe:", r.stdout.strip())
assert r.stdout.strip() == "¡Buenas, mi niño!" and "CANARIAS_SALUDO" not in os.environ

código 0: Hola desde otro proceso


código 3: a medias

check=True: CalledProcessError con código 3


timeout: cortado a los 0.5 s
El hijo recibe: ¡Buenas, mi niño!


## 10. `sys.argv` y códigos de salida

Escribir el programa `sumar.py`, que sume los números recibidos como argumentos (admitiendo coma decimal) y termine con código 0; con código 2 y un mensaje de uso si no recibe ninguno, y con código 1 si alguno no es un número. Los mensajes de error van a la salida de error. Probarlo con varias listas de argumentos.

In [12]:
%%writefile pruebas_os/sumar.py
"""Suma los números recibidos como argumentos."""
import os
import sys


def main(argumentos):
    if not argumentos:
        print(f"Uso: python {os.path.basename(sys.argv[0])} NÚMERO [NÚMERO ...]", file=sys.stderr)
        return 2
    numeros = []
    for texto in argumentos:
        try:
            numeros.append(float(texto.replace(",", ".")))
        except ValueError:
            print(f"Error: {texto!r} no es un número", file=sys.stderr)
            return 1
    print(f"{len(numeros)} {'número' if len(numeros) == 1 else 'números'}, suma = {sum(numeros):g}")
    return 0


if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))

Writing pruebas_os/sumar.py


In [13]:
SUMAR = os.path.join(CARPETA, "sumar.py")
casos = [["3", "4,5", "10"], ["-2.5"], [], ["7", "tres"]]
codigos = []
for argumentos in casos:
    r = ejecutar(SUMAR, *argumentos)
    codigos.append(r.returncode)
    canal, texto = ("stdout", r.stdout) if r.returncode == 0 else ("stderr", r.stderr)
    print(f"{' '.join(argumentos) or '(nada)':<12} código {r.returncode}  {canal}: {texto.strip()}")
assert codigos == [0, 0, 2, 1]

r = ejecutar("-c", "import sys; print(sys.argv)", "uno", "dos tres")
print("Con -c, sys.argv es:", r.stdout.strip())
assert r.stdout.strip() == "['-c', 'uno', 'dos tres']"

3 4,5 10     código 0  stdout: 3 números, suma = 17.5
-2.5         código 0  stdout: 1 número, suma = -2.5


(nada)       código 2  stderr: Uso: python sumar.py NÚMERO [NÚMERO ...]


7 tres       código 1  stderr: Error: 'tres' no es un número


Con -c, sys.argv es: ['-c', 'uno', 'dos tres']


## 11. Importar desde otra carpeta con `sys.path`

Crear el módulo `islas.py` en `pruebas_os/mis_modulos/`, con la superficie aproximada de cada isla y una función que devuelva la mayor. Comprobar que no se puede importar hasta añadir su carpeta a `sys.path`, usarlo y dejar `sys.path` como estaba. Hacer lo mismo en un proceso hijo con la variable de entorno `PYTHONPATH`.

In [14]:
import importlib
import textwrap

carpeta_modulos = os.path.join(CARPETA, "mis_modulos")
os.makedirs(carpeta_modulos, exist_ok=True)
Path(carpeta_modulos, "islas.py").write_text(textwrap.dedent('''\
    """Superficie aproximada de las islas Canarias en km²."""
    SUPERFICIE = {"Tenerife": 2034, "Fuerteventura": 1660, "Gran Canaria": 1560, "Lanzarote": 846,
                  "La Palma": 708, "La Gomera": 370, "El Hierro": 269}


    def mayor():
        return max(SUPERFICIE, key=SUPERFICIE.get)
'''), encoding="utf-8")

try:
    import islas
except ModuleNotFoundError as e:
    print("Antes:", e)

sys.path.insert(0, os.path.abspath(carpeta_modulos))
try:
    islas = importlib.import_module("islas")
    print("Después:", islas.mayor(), "desde", Path(islas.__file__).relative_to(Path.cwd()).as_posix())
    assert islas.mayor() == "Tenerife" and sum(islas.SUPERFICIE.values()) == 7447
finally:
    sys.path.remove(os.path.abspath(carpeta_modulos))
    sys.modules.pop("islas", None)
assert os.path.abspath(carpeta_modulos) not in sys.path

r = ejecutar("-c", "import islas; print(islas.mayor())", extra_env={"PYTHONPATH": carpeta_modulos})
print("Hijo con PYTHONPATH:", r.stdout.strip())
assert r.stdout.strip() == "Tenerife"

Antes: No module named 'islas'
Después: Tenerife desde pruebas_os/mis_modulos/islas.py
Hijo con PYTHONPATH: Tenerife


## 12. Límite de recursión

Consultar `sys.getrecursionlimit()` y comprobar que una suma recursiva con el triple de llamadas lanza `RecursionError`. Escribir un gestor de contexto que suba el límite temporalmente con `sys.setrecursionlimit` y lo restaure siempre, aunque haya un error, y calcular con él la misma suma.

In [15]:
from contextlib import contextmanager


def suma_hasta(n):
    return 0 if n == 0 else n + suma_hasta(n - 1)


@contextmanager
def limite_recursion(nuevo):
    anterior = sys.getrecursionlimit()
    sys.setrecursionlimit(nuevo)
    try:
        yield
    finally:
        sys.setrecursionlimit(anterior)


limite = sys.getrecursionlimit()
n = 3 * limite
print("Límite actual:", limite)
try:
    suma_hasta(n)
except RecursionError as e:
    print(f"suma_hasta({n}):", e)

with limite_recursion(n + 1_000):
    resultado = suma_hasta(n)
print(f"Con el límite subido: suma_hasta({n}) = {resultado}")
assert resultado == n * (n + 1) // 2 == sum(range(n + 1))
assert sys.getrecursionlimit() == limite

try:
    with limite_recursion(n + 1_000):
        raise ValueError("fallo dentro")
except ValueError:
    pass
assert sys.getrecursionlimit() == limite

Límite actual: 1000
suma_hasta(3000): maximum recursion depth exceeded
Con el límite subido: suma_hasta(3000) = 4501500


## 13. Borrar la carpeta de pruebas

Comprobar que `os.rmdir` no borra una carpeta que no está vacía, borrar un fichero con `os.remove` y una carpeta vacía con `os.rmdir`, y eliminar después todo `pruebas_os/` con `shutil.rmtree`.

In [16]:
try:
    os.rmdir(CARPETA)
except OSError as e:
    print("os.rmdir sobre una carpeta con contenido:", type(e).__name__)

oculto = os.path.join(CARPETA, "docs", ".oculto")
os.remove(oculto)
vacia = os.path.join(CARPETA, "vacia")
os.mkdir(vacia)
os.rmdir(vacia)
assert not os.path.exists(oculto) and not os.path.exists(vacia)

shutil.rmtree(CARPETA)
print(f"{CARPETA}/ borrada con todo su contenido:", not os.path.exists(CARPETA))
assert not os.path.exists(CARPETA)

os.rmdir sobre una carpeta con contenido:

 OSError
pruebas_os/ borrada con todo su contenido: True
